# Colonist Rating

This notebook calculates the Colonist players performance using [Trueskill](https://trueskill.org/) algorithm.

Game results are converted to head-to-head matches. For example, if player A, B, C play together and the result is A > B > C, then we will have three match results: A > B, A > C, B > C.

Trueskill assumes that the performance of a player in a match is **normally distributed**. The **mean** of the distribution represents the player's average skill, and the **standard deviation** represents the uncertainty in that skill estimate. After each match, the algorithm updates the skill estimates for all players involved based on the match outcome. The more matches a player participates in, the more accurate their skill estimate becomes, and the lower their standard deviation.

In [1]:
import pandas
from trueskill import Rating, rate_1vs1

In [2]:
# import the data
data = pandas.read_csv("data/record.csv")
data

,week,win,lose,draw
0,1,Roei,Shitian,False
1,1,Roei,Willow,False
2,1,Roei,Matt,False
3,1,Shitian,Willow,False
4,1,Shitian,Matt,False
5,1,Willow,Matt,True
6,2,Willow,Roei,False
7,2,Willow,Shitian,False
8,2,Willow,Matt,False
9,2,Roei,Shitian,False


In [3]:
# ratings of player list by alphabetical order
ratings = {
    'Claude': Rating(),
    'Matt': Rating(),
    'Roei': Rating(),
    'Shitian': Rating(),
    'Tom': Rating(),
    'Willow': Rating(),
}

In [5]:
# load the game result and update the player ratings
for index, row in data.iterrows():
    # get the head to head match results from the game result
    player1 = row['win']
    player2 = row['lose']
    draw = row['draw']
    # update the player ratings based on the match result
    ratings[player1], ratings[player2] = rate_1vs1(ratings[player1], ratings[player2], drawn=draw)


In [8]:
# build a table of player ratings
rating_table = pandas.DataFrame({
    'Player': ratings.keys(),
    'Rating': [rating.mu for rating in ratings.values()],
    'Uncertainty': [rating.sigma for rating in ratings.values()],
})
rating_table.sort_values(by='Rating', ascending=False, inplace=True)
rating_table.reset_index(drop=True, inplace=True)
rating_table.to_csv("data/rating.csv", index=False)

# round the rating table to 2 decimal places
rating_table = rating_table.round(2)
rating_table

,Player,Rating,Uncertainty
0,Roei,24.86,3.07
1,Willow,24.76,3.40
2,Shitian,24.70,2.90
3,Matt,22.21,2.96
4,Tom,17.66,5.99
5,Claude,17.31,4.08
